<table style="width:100%; border:none; background:transparent;">
  <tr style="border:none;">
    <td style="border:none; text-align:left; vertical-align:middle; width:22%;">
      <img src="./logos/EOSIAL_logo_Sapienza_white_background.bmp" alt="EOSIAL / Sapienza University of Rome" style="max-height:85px;">
    </td>
    <td style="border:none; text-align:center; vertical-align:middle;">
      <h1 style="margin:0; font-size:1.6em;">Fire Detection Challenge: Biebrza National Park Fires Detection with MTG-FCI Satellite Imagery</h1>
    </td>
    <td style="border:none; text-align:right; vertical-align:middle; width:22%;">
      <img src="./logos/agh_znk_wbr_rgb_150ppi.jpg" alt="AGH University of Krakow" style="max-height:85px;">
    </td>
  </tr>
</table>

**AGH University of Krakow — Aerospace Engineering**  
**Author:** Valerio Pampanoni ([valerio.pampanoni@uniroma1.it](mailto:valerio.pampanoni@uniroma1.it))  
**Affiliation:** Sapienza University of Rome — EOSIAL Lab

---

## Your Mission

On **April 20, 2025**, a major wildfire broke out in **Biebrza National Park**, Poland’s largest national park and one of Europe’s most important wetland habitats. The fire, fuelled by dry peatland and marshland vegetation, burned thousands of hectares and was monitored by the **Copernicus Emergency Management Service** (activation [EMSR801](https://mapping.emergency.copernicus.eu/news/wildfire-in-poland/)).

You have **MTG-FCI** satellite data: **13 timestamps** at **10-minute intervals** from **12:00 to 14:00 UTC** on April 20, 2025.

### Your objectives

1. **Build a working fire detection pipeline** using the provided functions
2. **Optimize the algorithm parameters** to detect the Biebrza fire **as early as possible**
3. **Minimise false alarms** — a detection system that flags everything is useless
4. **Visualise** your results with static plots and an interactive map
5. **Compute FRP timeseries** to track fire intensity over time

### The Rules

- Below you will find **ALL the Python functions** you need
- Each function is documented with a detailed docstring
- **YOUR JOB** is to figure out:
  - **In which order** to call the functions
  - **What hyperparameters** to use
  - **How to combine** results across multiple timestamps

> **The key challenge:** The default thresholds are designed for MODIS. MTG-FCI has different spectral response functions and resolution. You may need to **adjust parameters** — there is no single correct answer, it’s about finding the best **sensitivity vs. specificity** trade-off.

---

## What You Need to Know

### Why 3.8 µm for fire detection?

The **Planck function** peaks near 3.8 µm at ~800 K (typical fire temperature). At ~300 K (background), the 3.8 µm radiance is very low, so even a small sub-pixel fire produces a detectable signal. The **10.5 µm** channel peaks near 300 K and is much less sensitive to fire. The **difference** (MWIR – TIR) amplifies the fire signal.

### The MODIS Collection 5 algorithm (simplified)

**Stage 1 — Potential fire detection:**
- $T_{\text{MWIR}} > T_{\text{threshold}}$
- $T_{\text{MWIR}} - T_{\text{TIR}} > \Delta T_{\text{threshold}}$

**Stage 2 — Contextual confirmation:**
- **C1:** $T_{\text{MWIR}} > 360$ K → automatic confirm
- **C2:** $\Delta T > \overline{\Delta T}_{\text{bg}} + C_2 \cdot \sigma_{\Delta T}$
- **C3:** $\Delta T > \overline{\Delta T}_{\text{bg}} + C_3$
- **C4:** $T_{\text{MWIR}} > \overline{T}_{\text{MWIR,bg}} + C_4 \cdot \sigma_{\text{MWIR}}$

Confirmed if: **C1** OR (**C2** AND **C3** AND **C4**)

### Fire Radiative Power

$$\text{FRP [MW]} = A_{\text{pixel}} \times 4.34 \times 10^{-19} \times (T_{\text{fire}}^8 - T_{\text{bg}}^8)$$

---

## Part 1: The Toolkit

Below are all the functions you need. **Read the docstrings carefully.**

### 1.1 Data Access & I/O Functions

In [ ]:
def discover_timestamps(data_dir):
    """
    Scan the data directory and return a sorted list of datetime objects
    for which FDHSI, HRFI, and CLM zip files are all present.

    Parameters
    ----------
    data_dir : Path
        Directory containing the MTG zip archives.

    Returns
    -------
    list of datetime.datetime
        Sorted list of available observation times.
    """
    pattern = re.compile(r"MTG_FDHSI_POL_(\d{12})\.zip")
    timestamps = []
    for f in sorted(data_dir.glob("MTG_FDHSI_POL_*.zip")):
        m = pattern.match(f.name)
        if not m:
            continue
        dtstr = m.group(1)
        dtime = datetime.datetime.strptime(dtstr, "%Y%m%d%H%M")
        # Only include if the HRFI file is also present
        hrfi = data_dir / f"MTG_HRFI_POL_{dtstr}.zip"
        if hrfi.exists():
            timestamps.append(dtime)
    return timestamps


def extract_bands_from_zip(zip_path, band_names, out_dir):
    """
    Extract specific GeoTIFF bands from a zip archive into a directory.

    Parameters
    ----------
    zip_path : Path
        Path to the zip file.
    band_names : list of str
        Band filenames (without .tif extension) to extract.
    out_dir : Path
        Destination directory for extracted files.

    Returns
    -------
    dict
        {band_name: extracted_file_path_str}
    """
    extracted = {}
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.namelist():
            for band in band_names:
                if member.endswith(band + ".tif"):
                    out_path = out_dir / os.path.basename(member)
                    if not out_path.exists():
                        data = zf.read(member)
                        with open(out_path, "wb") as f:
                            f.write(data)
                    extracted[band] = str(out_path)
    return extracted


def read_band_as_masked_array(file_path, band_num=1):
    """
    Read a single band from a GeoTIFF into a NumPy masked array.

    The GDAL NoData value is used as the mask; NaN values are also masked.
    Accepts both regular file paths and GDAL virtual filesystem paths
    such as /vsizip/.

    Parameters
    ----------
    file_path : str
        Path to the GeoTIFF (or /vsizip/ path).
    band_num : int
        Band number to read (1-indexed).

    Returns
    -------
    numpy.ma.MaskedArray
    """
    ds = gdal.Open(file_path)
    if ds is None:
        raise ValueError(f"Could not open: {file_path}")
    band = ds.GetRasterBand(band_num)
    nodata = band.GetNoDataValue()
    arr = band.ReadAsArray().astype(np.float32)
    ds = None

    if nodata is not None and np.isnan(nodata):
        return ma.masked_invalid(arr)
    elif nodata is not None:
        return ma.masked_values(arr, nodata)
    else:
        return ma.masked_invalid(arr)


def get_geotransform(file_path):
    """Return the GDAL 6-element geotransform for a raster file."""
    ds = gdal.Open(file_path)
    gt = ds.GetGeoTransform()
    ds = None
    return gt


def get_projection_wkt(file_path):
    """Return the WKT projection string for a raster file."""
    ds = gdal.Open(file_path)
    wkt = ds.GetProjection()
    ds = None
    return wkt


def get_raster_extent(file_path):
    """Return [xmin, ymin, xmax, ymax] for a raster file."""
    ds = gdal.Open(file_path)
    gt = ds.GetGeoTransform()
    w, h = ds.RasterXSize, ds.RasterYSize
    ds = None
    xmin, ymax = gt[0], gt[3]
    xmax = xmin + w * gt[1] + h * gt[2]
    ymin = ymax + w * gt[4] + h * gt[5]
    return [xmin, ymin, xmax, ymax]


def get_lat_lon_grids(file_path, pixel_center=True):
    """
    Compute per-pixel longitude and latitude grids in WGS84 (EPSG:4326).

    Parameters
    ----------
    file_path : str
        Path to a georeferenced raster.
    pixel_center : bool
        If True, return coordinates for pixel centres.

    Returns
    -------
    lons, lats : numpy.ndarray
        2D arrays of the same shape as the raster.
    """
    ds = gdal.Open(file_path)
    gt = ds.GetGeoTransform()
    width, height = ds.RasterXSize, ds.RasterYSize
    src_wkt = ds.GetProjection()
    ds = None

    ox, pw, xr, oy, yr, ph = gt
    if pixel_center:
        ox += pw / 2.0
        oy += ph / 2.0

    cols, rows = np.meshgrid(np.arange(width), np.arange(height))
    x_nat = ox + cols * pw + rows * xr
    y_nat = oy + cols * yr + rows * ph

    src_srs = osr.SpatialReference()
    src_srs.ImportFromWkt(src_wkt)
    tgt_srs = osr.SpatialReference()
    tgt_srs.ImportFromEPSG(4326)
    tgt_srs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)

    transform = osr.CoordinateTransformation(src_srs, tgt_srs)
    pts = np.vstack((x_nat.ravel(), y_nat.ravel(), np.zeros(x_nat.size))).T
    transformed = np.array(transform.TransformPoints(pts))

    lons = transformed[:, 0].reshape(height, width)
    lats = transformed[:, 1].reshape(height, width)
    return lons, lats


def resample_to_target(source_path, target_shape, out_bounds, method="nearest"):
    """
    Resample a raster to a target grid using GDAL Warp (in-memory).

    Parameters
    ----------
    source_path : str
        Path to the source raster.
    target_shape : tuple
        (rows, cols) of the desired output.
    out_bounds : list
        [xmin, ymin, xmax, ymax] output bounds.
    method : str
        Resampling algorithm ('nearest', 'bilinear', etc.).

    Returns
    -------
    numpy.ndarray
    """
    methods = {"nearest": gdal.GRA_NearestNeighbour, "bilinear": gdal.GRA_Bilinear}
    ds = gdal.Open(source_path)
    mem_ds = gdal.Warp(
        "", ds, format="MEM",
        width=target_shape[1], height=target_shape[0],
        resampleAlg=methods[method], outputBounds=out_bounds,
    )
    arr = mem_ds.ReadAsArray()
    ds = None
    mem_ds = None
    return arr


def get_pixel_area_grid(geotransform, shape, src_crs_wkt, cache_file=None):
    """
    Compute per-pixel area (km²) using equal-area reprojection (EPSG:6933).

    Results are cached to a .npy file for repeated runs.

    Parameters
    ----------
    geotransform : tuple
        GDAL 6-element geotransform.
    shape : tuple
        (rows, cols).
    src_crs_wkt : str
        Source CRS as WKT.
    cache_file : str or Path, optional
        Path to .npy cache file.

    Returns
    -------
    numpy.ma.MaskedArray
        2D array of pixel areas in km².
    """
    if cache_file and os.path.exists(cache_file):
        return ma.masked_invalid(np.load(cache_file, allow_pickle=True))

    rows, cols = shape
    gt = geotransform
    ci, ri = np.meshgrid(np.arange(cols + 1), np.arange(rows + 1))
    X = gt[0] + ci * gt[1] + ri * gt[2]
    Y = gt[3] + ci * gt[4] + ri * gt[5]

    x_tl, y_tl = X[:-1, :-1].ravel(), Y[:-1, :-1].ravel()
    x_tr, y_tr = X[:-1, 1:].ravel(), Y[:-1, 1:].ravel()
    x_br, y_br = X[1:, 1:].ravel(), Y[1:, 1:].ravel()
    x_bl, y_bl = X[1:, :-1].ravel(), Y[1:, :-1].ravel()

    coords = np.array([[x_tl, y_tl], [x_tr, y_tr],
                       [x_br, y_br], [x_bl, y_bl]]).transpose(2, 0, 1)
    try:
        from shapely import polygons
        geoms = polygons(coords)
    except ImportError:
        from shapely.geometry import Polygon
        geoms = [Polygon(c) for c in coords]

    gs = gpd.GeoSeries(geoms, crs=src_crs_wkt)
    areas = gs.to_crs("EPSG:6933").area.values.reshape(rows, cols)
    area_km2 = ma.masked_array(areas, mask=~np.isfinite(areas)) / 1e6

    if cache_file:
        np.save(cache_file, area_km2.filled(np.nan))
    return area_km2

### 1.2 Data Cube & Masking Functions

In [ ]:
def build_data_cube(fdhsi_files, hrfi_files, target_shape, out_bounds):
    """
    Load all bands into a dictionary, resampling FDHSI bands to the
    HRFI resolution grid.

    Parameters
    ----------
    fdhsi_files : dict
        {band_name: file_path} for FDHSI bands.
    hrfi_files : dict
        {band_name: file_path} for HRFI bands.
    target_shape : tuple
        (rows, cols) of the HRFI grid.
    out_bounds : list
        [xmin, ymin, xmax, ymax] of the HRFI grid.

    Returns
    -------
    dict
        {tag: masked_array} for all bands.
    """
    cube = {}

    # HRFI bands are already at target resolution
    for band_name, tag in HRFI_BANDS.items():
        if band_name in hrfi_files:
            arr = read_band_as_masked_array(hrfi_files[band_name])
            np.place(arr, np.isnan(arr), 0.0)
            if hasattr(arr, 'fill_value') and arr.fill_value > 0.0:
                arr.fill_value = 0.0
            cube[tag] = arr

    # FDHSI bands need resampling to the HRFI grid
    for band_name, tag in FDHSI_BANDS.items():
        if band_name not in fdhsi_files:
            continue
        if tag == "NCC":
            arr_r = read_band_as_masked_array(fdhsi_files[band_name], band_num=1)
            arr_g = read_band_as_masked_array(fdhsi_files[band_name], band_num=2)
            arr_b = read_band_as_masked_array(fdhsi_files[band_name], band_num=3)
            cube[tag] = np.dstack([arr_r, arr_g, arr_b])
        else:
            resampled = resample_to_target(
                fdhsi_files[band_name], target_shape, out_bounds, method="nearest"
            )
            np.place(resampled, np.isnan(resampled), 0.0)
            cube[tag] = ma.masked_values(resampled, 0.0)

    return cube


def apply_exclusion_mask(cube, mask):
    """
    Mask out pixels in all cube bands (e.g. clouds, nighttime).

    Parameters
    ----------
    cube : dict
    mask : numpy.ndarray (bool)
        True = exclude this pixel.

    Returns
    -------
    dict
    """
    masked_cube = {}
    for key, arr in cube.items():
        if key == "NCC":
            masked_cube[key] = arr
            continue
        new_arr = arr.data.copy()
        fill = arr.fill_value if hasattr(arr, 'fill_value') else 0.0
        np.place(new_arr, mask, fill)
        masked_cube[key] = ma.masked_values(new_arr, fill)
    return masked_cube


def compute_cloud_mask(cube,
                       red_thresh=0.13,
                       nir13_thresh=0.01,
                       tir12_thresh=265.0):
    """
    Simple threshold-based cloud mask derived directly from the data cube.

    A pixel is flagged as cloudy if ANY of the following conditions is true:
      - Red reflectance (0.64 µm) > red_thresh       (bright cloud tops)
      - NIR 1.3 µm reflectance    > nir13_thresh      (cirrus detection)
      - TIR 12.3 µm BT           < tir12_thresh  [K]  (cold cloud tops)

    Parameters
    ----------
    cube : dict
        Data cube containing 'RED_REF', 'NIR_13_REF', and 'TIR_12_BT'.
    red_thresh : float
        Red reflectance threshold (default 0.13).
    nir13_thresh : float
        NIR 1.3 µm reflectance threshold (default 0.01).
    tir12_thresh : float
        TIR 12.3 µm brightness temperature threshold in K (default 265.0).

    Returns
    -------
    numpy.ndarray (bool)
        True = cloudy pixel.
    """
    mask_red  = cube["RED_REF"].filled(0)    > red_thresh
    mask_nir  = cube["NIR_13_REF"].filled(0) > nir13_thresh
    mask_tir  = cube["TIR_12_BT"].filled(999) < tir12_thresh

    return mask_red | mask_nir | mask_tir

### 1.3 Fire Detection Functions

In [ ]:
def detect_potential_hotspots(cube, mwir_thresh, diff_thresh):
    """
    MODIS Collection 5 potential hotspot detection (daytime).

    A pixel is flagged if BOTH conditions are true:
      (1) MWIR brightness temperature > mwir_thresh
      (2) MWIR – TIR brightness temperature difference > diff_thresh

    Parameters
    ----------
    cube : dict
        Data cube with keys "MWIR_BT" and "TIR_11_BT".
    mwir_thresh : float
        MWIR BT threshold [K].
    diff_thresh : float
        MWIR–TIR difference threshold [K].

    Returns
    -------
    numpy.ndarray (bool)
    """
    mwir = cube["MWIR_BT"]
    tir  = cube["TIR_11_BT"]
    return np.logical_and(mwir > mwir_thresh, (mwir - tir) > diff_thresh)


def compute_background_stats(image, fire_mask, window_size):
    """
    Compute windowed background mean and standard deviation,
    excluding fire pixels from the statistics.

    Uses FFT-based convolution for efficiency.

    Parameters
    ----------
    image : numpy.ma.MaskedArray
    fire_mask : numpy.ndarray (bool)
        True = fire pixel (excluded from background).
    window_size : int

    Returns
    -------
    bg_mean, bg_std : numpy.ndarray
    """
    data  = image.filled(0.0).astype(float)
    valid = (~(ma.getmaskarray(image) | fire_mask)).astype(float)
    kernel = np.ones((window_size, window_size), dtype=float)

    def fft_conv(a, b):
        s = (a.shape[0] + b.shape[0] - 1, a.shape[1] + b.shape[1] - 1)
        out = np.fft.irfftn(np.fft.rfftn(a, s=s) * np.fft.rfftn(b, s=s), s=s)
        pm, pn = (b.shape[0]-1)//2, (b.shape[1]-1)//2
        return out[pm:pm+a.shape[0], pn:pn+a.shape[1]]

    count = fft_conv(valid, kernel)
    with np.errstate(divide="ignore", invalid="ignore"):
        sum_x  = fft_conv(data * valid, kernel)
        sum_x2 = fft_conv((data**2) * valid, kernel)
        bg_mean = np.where(count > 0, sum_x / count, np.nan)
        bg_var  = np.where(count > 0, sum_x2 / count - bg_mean**2, np.nan)
        bg_std  = np.sqrt(np.maximum(bg_var, 0.0))
    return bg_mean, bg_std


def confirm_hotspots(cube, pot_fire_mask, window_size, c1, c2, c3, c4):
    """
    Contextual confirmation of potential hotspots (MODIS C5 algorithm).

    A pixel is confirmed if:
      C1: MWIR BT > c1                       (absolute — auto-confirm)
      OR all of:
      C2: MWIR–TIR diff > bg_mean_diff + c2 × bg_std_diff
      C3: MWIR–TIR diff > bg_mean_diff + c3
      C4: MWIR BT        > bg_mean_mwir + c4 × bg_std_mwir

    Parameters
    ----------
    cube : dict
    pot_fire_mask : numpy.ndarray (bool)
    window_size : int
    c1, c2, c3, c4 : float  (see configuration cell)

    Returns
    -------
    numpy.ndarray (bool)
    """
    mwir = cube["MWIR_BT"]
    tir  = cube["TIR_11_BT"]
    diff = mwir - tir

    mir_mean,  mir_std  = compute_background_stats(mwir, pot_fire_mask, window_size)
    diff_mean, diff_std = compute_background_stats(
        ma.masked_array(diff if not hasattr(diff, "filled") else diff,
                        mask=ma.getmaskarray(mwir)),
        pot_fire_mask, window_size,
    )

    mwir_vals = np.where(pot_fire_mask, mwir.filled(0), 0)
    diff_vals = np.where(pot_fire_mask, diff.filled(0) if hasattr(diff, "filled") else diff, 0)

    confirmed = np.logical_or(
        mwir_vals > c1,
        np.logical_and(
            diff_vals > diff_mean + c2 * diff_std,
            np.logical_and(
                diff_vals > diff_mean + c3,
                mwir_vals > mir_mean  + c4 * mir_std,
            ),
        ),
    )
    return np.logical_and(confirmed, pot_fire_mask)


def compute_frp_modis(mwir_bt_fire, mwir_bt_bg, pixel_area_km2):
    """
    Fire Radiative Power using the MODIS 8th-power approximation.

    FRP [MW] = pixel_area [km²] × 4.34×10⁻¹⁹ × (T_fire⁸ – T_bg⁸)

    Parameters
    ----------
    mwir_bt_fire : numpy.ndarray  (K)
    mwir_bt_bg   : numpy.ndarray  (K, background BT)
    pixel_area_km2 : numpy.ndarray  (km²)

    Returns
    -------
    numpy.ndarray  FRP in MW per pixel.
    """
    return pixel_area_km2 * 4.34e-19 * (mwir_bt_fire**8 - mwir_bt_bg**8)


print("Fire detection functions defined.")

### 1.4 Visualization Functions

In [ ]:
def plot_band(arr, title, cmap="gray", vmin=None, vmax=None, cbar_label=None, figsize=(10, 8)):
    """Plot a 2D array with colorbar."""
    fig, ax = plt.subplots(figsize=figsize, dpi=150)
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontweight="bold", fontsize=13)
    ax.axis("off")
    plt.colorbar(im, ax=ax, label=cbar_label or "", shrink=0.7)
    plt.tight_layout()
    plt.show()
    return fig

def plot_fire_overlay(background, fire_mask, title, frp_values=None, vmin=270, vmax=320, figsize=(12, 10)):
    """Overlay fire detections on a background image."""
    fig, ax = plt.subplots(figsize=figsize, dpi=150)
    ax.imshow(background, cmap="gray", vmin=vmin, vmax=vmax)
    if frp_values is not None and np.any(fire_mask):
        fire_display = np.where(fire_mask, frp_values, np.nan)
        im = ax.imshow(np.ma.masked_invalid(fire_display), cmap="YlOrRd", vmin=0,
            vmax=max(float(np.nanpercentile(frp_values[fire_mask], 95)), 1.0))
        plt.colorbar(im, ax=ax, label="FRP [MW]", shrink=0.7)
    else:
        overlay = np.ma.masked_where(~fire_mask, np.ones_like(fire_mask, float))
        ax.imshow(overlay, cmap="autumn_r", alpha=0.85)
    ax.set_title(title, fontweight="bold", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    return fig

def plot_ncc(ncc_arr, title, figsize=(12, 10)):
    """Plot natural colour composite with contrast stretching."""
    ncc = ncc_arr.copy().astype(float)
    for i in range(3):
        band = ncc[:, :, i]
        valid = band[band > 0]
        if valid.size:
            p2, p98 = np.nanpercentile(valid, [2, 98])
            ncc[:, :, i] = np.clip((band - p2) / max(p98 - p2, 1e-9), 0, 1)
    fig, ax = plt.subplots(figsize=figsize, dpi=150)
    ax.imshow(ncc)
    ax.set_title(title, fontweight="bold", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    return fig

def plot_frp_timeseries(timestamps, frp_total, frp_biebrza, figsize=(13, 5)):
    """Plot FRP timeseries."""
    fig, axes = plt.subplots(1, 2, figsize=figsize, dpi=150)
    for ax, vals, label, color, title in [
        (axes[0], frp_total, "Total FRP [MW]", "#d62728", "Total Scene FRP"),
        (axes[1], frp_biebrza, "Biebrza FRP [MW]", "#1f77b4", "Biebrza NP FRP"),
    ]:
        ax.plot(timestamps, vals, marker="o", linewidth=2, color=color,
                markerfacecolor="white", markeredgewidth=2)
        ax.fill_between(timestamps, vals, alpha=0.15, color=color)
        ax.set_xlabel("Time (UTC)"); ax.set_ylabel(label)
        ax.set_title(title, fontweight="bold")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        ax.grid(axis="y", linestyle="--", alpha=0.5); ax.set_ylim(bottom=0)
    plt.tight_layout(); plt.show()
    return fig

def plot_cumulative_frp(timestamps, frp_biebrza, figsize=(10, 5)):
    """Plot cumulative FRE."""
    fre = [f * 600 for f in frp_biebrza]
    cumfre = np.cumsum(fre)
    fig, ax = plt.subplots(figsize=figsize, dpi=150)
    ax.plot(timestamps, cumfre / 1e3, marker="o", linewidth=2, color="#2ca02c",
            markerfacecolor="white", markeredgewidth=2)
    ax.fill_between(timestamps, cumfre / 1e3, alpha=0.15, color="#2ca02c")
    ax.set_xlabel("Time (UTC)"); ax.set_ylabel("Cumulative FRE [GJ]")
    ax.set_title("Cumulative FRE — Biebrza NP", fontweight="bold")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.grid(axis="y", linestyle="--", alpha=0.5); ax.set_ylim(bottom=0)
    plt.tight_layout(); plt.show()
    return fig

def build_interactive_map(gdf_all, bbox):
    """Build leafmap interactive map."""
    from shapely.geometry import box
    m = leafmap.Map(center=[52.5, 20.5], zoom=7)
    m.add_basemap("Esri.WorldImagery")
    if not gdf_all.empty:
        m.add_gdf(gdf_all, layer_name="Fire Detections",
            style={"color": "red", "fillColor": "orange", "fillOpacity": 0.65, "radius": 4, "weight": 1})
    biebrza_poly = gpd.GeoDataFrame(
        geometry=[box(bbox["lon_min"], bbox["lat_min"], bbox["lon_max"], bbox["lat_max"])], crs="EPSG:4326")
    m.add_gdf(biebrza_poly, layer_name="Biebrza NP (AOI)",
        style={"color": "#2ca02c", "fillOpacity": 0.04, "weight": 2})
    return m

### 1.5 Helper Function

In [ ]:
def cleanup_temp(temp_dir):
    """Remove extracted .tif files from a per-timestamp temp subdirectory."""
    for f in temp_dir.glob("*.tif"):
        f.unlink()
    try:
        temp_dir.rmdir()
    except OSError:
        pass

### 1.6 Band Mapping Reference

| Product | Band Key | Tag | Wavelength | Description |
|---------|----------|-----|------------|-------------|
| FDHSI | `vis_06` | `RED_REF` | 0.64 µm | Red reflectance |
| FDHSI | `vis_08` | `NIR_08_REF` | 0.86 µm | NIR reflectance |
| FDHSI | `nir_13` | `NIR_13_REF` | 1.3 µm | NIR (cirrus) reflectance |
| FDHSI | `nir_22` | `SWIR_22_REF` | 2.2 µm | SWIR reflectance |
| FDHSI | `ir_123` | `TIR_12_BT` | 12.3 µm | Brightness temperature |
| FDHSI | `natural_color` | `NCC` | — | Natural colour composite |
| HRFI | `ir_38` | `MWIR_BT` | 3.8 µm | **Brightness temperature (KEY)** |
| HRFI | `ir_38_rad` | `MWIR_RAD` | 3.8 µm | Radiance |
| HRFI | `ir_105` | `TIR_11_BT` | 10.5 µm | Brightness temperature |

---

## Part 2: The Challenge

### Deliverables

1. **Fire detection summary** — number of fires and FRP per timestamp
2. **Static visualizations** — MWIR BT, MWIR–TIR difference, fire overlay
3. **FRP timeseries plots** — total scene and Biebrza FRP over time
4. **Cumulative FRE plot**
5. **Interactive leafmap** with all detections
6. **GeoJSON file** with all fire detections

### Key questions to answer

- At what time do you **first detect** the Biebrza fire?
- How many **false alarms** does your algorithm produce?
- What happens when you make the cloud mask more/less aggressive?
- What happens when you lower the MWIR threshold from 310 K to 305 K?
- How sensitive are results to the contextual confirmation parameters?

<details>
<summary><b>Hint 1:</b> What is the correct processing order for ONE timestamp?</summary>

1. Extract bands from zip archives (FDHSI + HRFI)
2. Determine target grid from an HRFI band (shape + extent)
3. Build data cube (loads all bands, resamples FDHSI to HRFI grid)
4. Compute solar zenith angle → create day/night mask (SZA > 85° = night)
5. Compute cloud mask
6. Combine night + cloud into an exclusion mask
7. Apply exclusion mask to the data cube
8. Detect potential hotspots
9. Confirm hotspots using contextual tests
10. Compute FRP for confirmed pixels
11. Build GeoDataFrame with results
12. Clean up temp files
</details>

<details>
<summary><b>Hint 2:</b> How to handle multiple timestamps?</summary>

Loop over all timestamps from `discover_timestamps()`. For each one, run the full pipeline and collect results. Use a shared pixel-area cache file to avoid recomputing pixel areas every time.

Store per-timestamp FRP totals for the timeseries plots. Separate Biebrza-specific fires using bounding box coordinates.
</details>

<details>
<summary><b>Hint 3:</b> Starting parameter values</summary>

| Parameter | Starting Value | What it controls |
|-----------|---------------|------------------|
| `red_thresh` | 0.13 | Cloud mask: visible brightness |
| `nir13_thresh` | 0.01 | Cloud mask: cirrus detection |
| `tir12_thresh` | 265.0 K | Cloud mask: cold cloud tops |
| `mwir_thresh` | 310.0 K | Potential fire: min MWIR BT |
| `diff_thresh` | 10.0 K | Potential fire: min MWIR–TIR diff |
| `C1` | 360.0 K | Confirmation: absolute MWIR |
| `C2` | 3.0 | Confirmation: diff std multiplier |
| `C3` | 5.5 K | Confirmation: diff absolute offset |
| `C4` | 2.5 | Confirmation: MWIR std multiplier |
| `window_size` | 21 px | Background statistics window |
</details>

<details>
<summary><b>Hint 4:</b> What if you get zero detections?</summary>

Check step by step:
1. Print the MWIR max after cloud masking — is the fire pixel being masked?
2. Print the number of potential hotspots — if zero, lower your MWIR threshold
3. If potentials exist but no confirmations, check background stats at the hottest pixel
4. Try `C1 = 320 K` temporarily to force-confirm and verify potential pixels are real
</details>

<details>
<summary><b>Hint 5:</b> Cloud mask optimization</summary>

The cloud mask is a balancing act:
- **Too aggressive** → masks fire pixels (fires are bright in visible and warm!)
- **Too lenient** → clouds cause false fire detections

The NIR 1.3 µm channel is particularly tricky: it detects cirrus, but fire aerosol plumes can also trigger it. Try values between 0.005 and 0.02.

Print the percentage masked by each individual test to understand their impact.
</details>

---

## Part 3: Your Workspace

Use the cells below to build your fire detection pipeline. Good luck!

In [ ]:
# ========================================================================
# Step 0: Setup
# ========================================================================
import datetime
import os
import re
import zipfile
from pathlib import Path

import numpy as np
import numpy.ma as ma
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import geopandas as gpd
import pandas as pd
from osgeo import gdal, osr
from pyorbital.astronomy import sun_zenith_angle
import leafmap

gdal.UseExceptions()

BASE_DIR = Path(".").resolve().parent
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
TEMP_DIR = BASE_DIR / "temp"

OUTPUT_DIR.mkdir(exist_ok=True)
TEMP_DIR.mkdir(exist_ok=True)

In [ ]:
# ========================================================================
# Step 1: Configuration
# ========================================================================

# ── MTG-FCI band mapping (pre-compiled, do not change) ───────────────────
FDHSI_BANDS = {
    "vis_06":        "RED_REF",      # 0.64 µm — red reflectance
    "vis_08":        "NIR_08_REF",   # 0.86 µm — NIR reflectance
    "nir_13":        "NIR_13_REF",   # 1.30 µm — cirrus/cloud NIR
    "nir_22":        "SWIR_22_REF",  # 2.20 µm — SWIR reflectance
    "ir_123":        "TIR_12_BT",    # 12.3 µm — brightness temperature
    "natural_color": "NCC",          # RGB natural colour composite
}
HRFI_BANDS = {
    "ir_38":     "MWIR_BT",    # 3.8 µm — brightness temperature (primary fire band)
    "ir_38_rad": "MWIR_RAD",   # 3.8 µm — radiance (used for FRP)
    "ir_105":    "TIR_11_BT",  # 10.5 µm — brightness temperature (background)
}

# ── Detection thresholds — start here, tune in Step 5 ───────────────────
MWIR_THRESHOLD          = 310.0  # K  — min MWIR BT to flag as potential fire
MWIR_TIR_DIFF_THRESHOLD =  10.0  # K  — min MWIR–TIR BT difference

# ── MODIS Collection 5 contextual confirmation coefficients ─────────────
CONTEXT_WINDOW =  21     # square window side length [pixels]
C1             = 360.0   # absolute MWIR BT for auto-confirmation [K]
C2             =   3.0   # MWIR–TIR diff > bg_mean + C2 × bg_std
C3             =   5.5   # MWIR–TIR diff > bg_mean + C3  [K]
C4             =   2.5   # MWIR BT       > bg_mean + C4 × bg_std

# ── Area of interest ─────────────────────────────────────────────────────
BIEBRZA_BBOX = {
    "lon_min": 22.4, "lon_max": 23.6,
    "lat_min": 53.3, "lat_max": 53.75,
}

print("Configuration ready.")
print(f"  MWIR threshold:     {MWIR_THRESHOLD} K")
print(f"  MWIR–TIR threshold: {MWIR_TIR_DIFF_THRESHOLD} K")
print(f"  Confirmation:       C1={C1}, C2={C2}, C3={C3}, C4={C4}")

In [ ]:
# ========================================================================
# Step 2: Discover available timestamps
# ========================================================================
# discover_timestamps(data_dir) scans for matching FDHSI+HRFI zip pairs
# and returns a sorted list of datetime objects.

timestamps = discover_timestamps(DATA_DIR)

print(f"Found {len(timestamps)} timestamps:")
for t in timestamps:
    print(f"  {t.strftime('%Y-%m-%d %H:%M UTC')}")

In [ ]:
# ========================================================================
# Step 3: Process a SINGLE timestamp  (debug here before the full loop)
# ========================================================================

# Choose your starting timestamp — index 6 = 13:00 UTC (strong fire signal)
dtime    = timestamps[6]   # ← try other indices after Step 5
dtstr    = dtime.strftime("%Y%m%d%H%M")
ts_label = dtime.strftime("%Y-%m-%d %H:%M UTC")
print(f"Processing: {ts_label}")

# Locate zip archives and create a temp extraction folder
fdhsi_zip = DATA_DIR / f"MTG_FDHSI_POL_{dtstr}.zip"
hrfi_zip  = DATA_DIR / f"MTG_HRFI_POL_{dtstr}.zip"
ts_temp   = TEMP_DIR / dtstr
ts_temp.mkdir(exist_ok=True)

# Extract band GeoTIFFs from the zip archives
#   extract_bands_from_zip(zip_path, band_names, out_dir) → {band_name: file_path}
fdhsi_files = extract_bands_from_zip(fdhsi_zip, list(FDHSI_BANDS.keys()), ts_temp)
hrfi_files  = extract_bands_from_zip(hrfi_zip,  list(HRFI_BANDS.keys()),  ts_temp)

# Get reference grid parameters from the HRFI file (≈1 km resolution)
ref_file     = list(hrfi_files.values())[0]
target_shape = read_band_as_masked_array(ref_file).shape
out_bounds   = get_raster_extent(ref_file)

# Build the multi-band data cube (FDHSI bands resampled to HRFI resolution)
#   build_data_cube(fdhsi_files, hrfi_files, target_shape, out_bounds) → {tag: masked_array}
cube = build_data_cube(fdhsi_files, hrfi_files, target_shape, out_bounds)
print(f"Cube bands: {list(cube.keys())} | shape: {cube['MWIR_BT'].shape}")

# Compute per-pixel lat/lon grids and solar zenith angle for nighttime masking
#   get_lat_lon_grids(ref_file, pixel_center=True) → lons, lats (2D arrays)
#   sun_zenith_angle(dtime, lons, lats)            → sza (degrees)
lons, lats = get_lat_lon_grids(ref_file, pixel_center=True)
sza        = sun_zenith_angle(dtime, lons, lats)
mask_night = sza > 85.0

# Compute cloud mask (True = cloudy) and apply all exclusion masks
#   compute_cloud_mask(cube) → bool array
#   apply_exclusion_mask(cube, exclusion) → masked cube
cloud_mask = compute_cloud_mask(cube)
exclusion  = np.logical_or(mask_night, cloud_mask)
cube_day   = apply_exclusion_mask(cube, exclusion)
print(f"Excluded pixels (night + cloud): {int(np.sum(exclusion)):,} / {exclusion.size:,}")

# ── 3a: Detect potential hotspots ───────────────────────────────────────
# Flag pixels where: MWIR_BT > MWIR_THRESHOLD  AND  (MWIR–TIR) > MWIR_TIR_DIFF_THRESHOLD
#   detect_potential_hotspots(cube_day, mwir_thresh, diff_thresh) → bool array
# TODO: uncomment ↓
# pot_fires = detect_potential_hotspots(cube_day, MWIR_THRESHOLD, MWIR_TIR_DIFF_THRESHOLD)
# print(f"Potential hotspots: {int(np.sum(pot_fires))}")

# ── 3b: Contextual confirmation ─────────────────────────────────────────
# Test each candidate against local background statistics (C1–C4 criteria)
#   confirm_hotspots(cube_day, pot_fires, CONTEXT_WINDOW, c1=C1, c2=C2, c3=C3, c4=C4)
# TODO: uncomment ↓
# confirmed = confirm_hotspots(
#     cube_day, pot_fires, CONTEXT_WINDOW,
#     c1=C1, c2=C2, c3=C3, c4=C4,
# )
# print(f"Confirmed fires: {int(np.sum(confirmed))}")

# ── 3c: Fire Radiative Power (FRP) ──────────────────────────────────────
# FRP [MW] = pixel_area [km²] × 4.34×10⁻¹⁹ × (T_fire⁸ – T_background⁸)
#   get_pixel_area_grid(gt, shape, wkt, cache_file=...) → area array [km²]
#   compute_background_stats(cube_day["MWIR_BT"], confirmed, CONTEXT_WINDOW) → mean, std
#   compute_frp_modis(mwir_bt_fire, mwir_bt_bg, pixel_area_km2) → FRP [MW] per pixel
# TODO: uncomment ↓
# pixel_area_cache = str(TEMP_DIR / "pixel_area_POL_MTG.npy")
# pixel_area       = get_pixel_area_grid(
#     get_geotransform(ref_file), target_shape, get_projection_wkt(ref_file),
#     cache_file=pixel_area_cache,
# )
# mir_bg_mean, _   = compute_background_stats(cube_day["MWIR_BT"], confirmed, CONTEXT_WINDOW)
# frp = np.where(
#     confirmed,
#     compute_frp_modis(cube_day["MWIR_BT"].filled(0), mir_bg_mean, pixel_area.filled(0)),
#     0.0,
# )
# total_frp = float(np.nansum(frp[confirmed])) if np.any(confirmed) else 0.0
# print(f"Total FRP: {total_frp:.1f} MW")

cleanup_temp(ts_temp)

In [ ]:
# ========================================================================
# Step 4: Examine your intermediate results
# ========================================================================
# Prerequisite: Step 3 completed (at minimum through Step 3a for fire plots).

# ── MWIR BT — fires appear as bright spots in the inferno colormap ─────
plot_band(
    cube["MWIR_BT"].filled(np.nan),
    f"MWIR Brightness Temperature (3.8 µm) — {ts_label}",
    cmap="inferno", vmin=270, vmax=340, cbar_label="BT [K]",
)

# ── MWIR – TIR difference — amplifies the fire signal ──────────────────
diff = (cube["MWIR_BT"] - cube["TIR_11_BT"]).filled(np.nan)
plot_band(
    diff, f"MWIR – TIR Difference — {ts_label}",
    cmap="RdBu_r", vmin=-15, vmax=30, cbar_label="ΔT [K]",
)

# ── Cloud mask ────────────────────────────────────────────────────────────
plot_band(cloud_mask.astype(float), "Cloud Mask  (1 = cloudy)", cmap="gray_r", vmin=0, vmax=1)

# ── Confirmed fires overlay (requires Step 3b + 3c) ──────────────────────
# TODO: uncomment once confirmed and frp are defined
# plot_fire_overlay(
#     cube["MWIR_BT"].filled(np.nan), confirmed,
#     f"Confirmed Fires (n={int(np.sum(confirmed))}) — {ts_label}",
#     frp_values=frp,
# )

# ── Natural colour composite ──────────────────────────────────────────────
# if "NCC" in cube:
#     plot_ncc(cube["NCC"], f"Natural Colour Composite — {ts_label}")

# ── Quick diagnostics ────────────────────────────────────────────────────
print(f"Max MWIR BT:  {float(np.nanmax(cube['MWIR_BT'].filled(np.nan))):.1f} K")
print(f"Cloud pixels: {int(np.sum(cloud_mask)):,}")
print(f"Night pixels: {int(np.sum(mask_night)):,}")
# print(f"Confirmed:    {int(np.sum(confirmed))}")
# if np.any(confirmed): print(f"Max FRP:      {float(np.nanmax(frp[confirmed])):.1f} MW")

In [ ]:
# ========================================================================
# Step 5: Tune your thresholds
# ========================================================================
# Go back to Step 1 (config cell), adjust the values, then re-run Steps 3 & 4.
# Goal: detect the Biebrza fire as early as possible with few false alarms.

# ── Suggested experiments ───────────────────────────────────────────────
#
# Detection thresholds            Default    Try
#   MWIR_THRESHOLD                310 K   →  300, 305, 320 K
#   MWIR_TIR_DIFF_THRESHOLD        10 K   →  5, 8, 15 K
#
# Confirmation coefficients (loosen → more detections, tighten → fewer false alarms)
#   C1   360 K  →  try 340, 350 K
#   C2   3.0    →  try 2.5, 2.0
#   C3   5.5 K  →  try 4.0, 3.0 K
#   C4   2.5    →  try 2.0, 1.5
#
# Cloud mask aggressiveness (swap in Step 3 where compute_cloud_mask is called):
#   compute_cloud_mask(cube)                           ← default
#   compute_cloud_mask(cube, red_thresh=0.20)          ← stricter bright-cloud filter
#   compute_cloud_mask(cube, nir13_thresh=0.005)       ← stricter cirrus filter
#   compute_cloud_mask(cube, tir12_thresh=260.0)       ← looser cold-cloud filter

# ── Challenge questions ──────────────────────────────────────────────────
# 1. What is the earliest timestamp at which you detect the Biebrza fire?
# 2. At what threshold does your false alarm rate become unacceptable?
# 3. How does loosening the cloud mask affect your fire pixel count?

print("Adjust thresholds in Step 1, then re-run Steps 3 – 4.")

In [ ]:
# ========================================================================
# Step 6: Process ALL timestamps
# ========================================================================

all_gdfs         = []
frp_total_ts     = []
frp_biebrza_ts   = []
per_ts_data      = {}   # stores (cube, confirmed, frp, lons, lats) per dtime
pixel_area_cache = str(TEMP_DIR / "pixel_area_POL_MTG.npy")

print(f"{"Time":>8} | {"Fires":>6} | {"Total FRP":>10} | {"Biebrza FRP":>12}")
print("-" * 48)

for dtime in timestamps:
    dtstr     = dtime.strftime("%Y%m%d%H%M")
    fdhsi_zip = DATA_DIR / f"MTG_FDHSI_POL_{dtstr}.zip"
    hrfi_zip  = DATA_DIR / f"MTG_HRFI_POL_{dtstr}.zip"
    ts_temp   = TEMP_DIR / dtstr
    ts_temp.mkdir(exist_ok=True)

    # Extract bands and build cube
    fdhsi_files = extract_bands_from_zip(fdhsi_zip, list(FDHSI_BANDS.keys()), ts_temp)
    hrfi_files  = extract_bands_from_zip(hrfi_zip,  list(HRFI_BANDS.keys()),  ts_temp)
    ref_file     = list(hrfi_files.values())[0]
    target_shape = read_band_as_masked_array(ref_file).shape
    out_bounds   = get_raster_extent(ref_file)
    cube         = build_data_cube(fdhsi_files, hrfi_files, target_shape, out_bounds)

    # Masks
    lons, lats = get_lat_lon_grids(ref_file, pixel_center=True)
    sza        = sun_zenith_angle(dtime, lons, lats)
    exclusion  = np.logical_or(sza > 85.0, compute_cloud_mask(cube))
    cube_day   = apply_exclusion_mask(cube, exclusion)

    # TODO: replace the two lines below with your detect + confirm calls from Step 3
    pot_fires = detect_potential_hotspots(cube_day, MWIR_THRESHOLD, MWIR_TIR_DIFF_THRESHOLD)
    confirmed = confirm_hotspots(cube_day, pot_fires, CONTEXT_WINDOW, c1=C1, c2=C2, c3=C3, c4=C4)

    # FRP
    pixel_area   = get_pixel_area_grid(
        get_geotransform(ref_file), target_shape, get_projection_wkt(ref_file),
        cache_file=pixel_area_cache,
    )
    mir_bg_mean, _ = compute_background_stats(cube_day["MWIR_BT"], confirmed, CONTEXT_WINDOW)
    frp = np.where(
        confirmed,
        compute_frp_modis(cube_day["MWIR_BT"].filled(0), mir_bg_mean, pixel_area.filled(0)),
        0.0,
    )

    # Per-timestamp stats
    total_frp = float(np.nansum(frp[confirmed])) if np.any(confirmed) else 0.0
    in_bbox   = (
        (lons >= BIEBRZA_BBOX["lon_min"]) & (lons <= BIEBRZA_BBOX["lon_max"]) &
        (lats >= BIEBRZA_BBOX["lat_min"]) & (lats <= BIEBRZA_BBOX["lat_max"])
    )
    biebrza_frp = float(np.nansum(frp[confirmed & in_bbox]))
    frp_total_ts.append(total_frp)
    frp_biebrza_ts.append(biebrza_frp)
    per_ts_data[dtime] = (cube, confirmed, frp, lons, lats)

    # Build GeoDataFrame for confirmed fire pixels
    if np.any(confirmed):
        df = pd.DataFrame({
            "LATITUDE":     lats[confirmed],
            "LONGITUDE":    lons[confirmed],
            "ACQ_DATE":     dtime.strftime("%Y-%m-%d"),
            "ACQ_TIME":     dtime.strftime("%H:%M"),
            "DATETIME":     dtime,
            "SATELLITE":    "MTG-1",
            "INSTRUMENT":   "FCI",
            "BRIGHT_MIR":   np.round(cube_day["MWIR_BT"].filled(0)[confirmed],   2),
            "BRIGHT_TIR":   np.round(cube_day["TIR_11_BT"].filled(0)[confirmed], 2),
            "MIR_TIR_DIFF": np.round(
                cube_day["MWIR_BT"].filled(0)[confirmed] -
                cube_day["TIR_11_BT"].filled(0)[confirmed], 2),
            "FRP_MW":       np.round(frp[confirmed], 2),
        })
        gdf = gpd.GeoDataFrame(
            df, geometry=gpd.points_from_xy(df.LONGITUDE, df.LATITUDE), crs="EPSG:4326"
        )
        all_gdfs.append(gdf)

    print(f"{dtime.strftime("%H:%M UTC"):>8} | {int(np.sum(confirmed)):>6} |"
          f" {total_frp:>9.1f} MW | {biebrza_frp:>11.1f} MW")
    cleanup_temp(ts_temp)

gdf_all = (
    gpd.GeoDataFrame(pd.concat(all_gdfs, ignore_index=True), crs="EPSG:4326")
    if all_gdfs else gpd.GeoDataFrame()
)
print(f"\nTotal fire detections: {len(gdf_all)}")

In [ ]:
# ========================================================================
# Step 7: Visualizations
# ========================================================================
# per_ts_data stores (cube, confirmed, frp, lons, lats) for every timestamp.
# Pick any timestamp with fires to visualise.

# ── Choose a timestamp to inspect ───────────────────────────────────────
vis_dtime = timestamps[6]   # 13:00 UTC — change to explore others
cube_v, conf_v, frp_v, lons_v, lats_v = per_ts_data[vis_dtime]
ts_label_v = vis_dtime.strftime("%Y-%m-%d %H:%M UTC")

# ── Static band plots ────────────────────────────────────────────────────
plot_band(
    cube_v["MWIR_BT"].filled(np.nan),
    f"MWIR Brightness Temperature (3.8 µm) — {ts_label_v}",
    cmap="inferno", vmin=270, vmax=340, cbar_label="BT [K]",
)

diff_v = (cube_v["MWIR_BT"] - cube_v["TIR_11_BT"]).filled(np.nan)
plot_band(
    diff_v, f"MWIR – TIR Difference — {ts_label_v}",
    cmap="RdBu_r", vmin=-15, vmax=30, cbar_label="ΔT [K]",
)

# ── Fire overlay ─────────────────────────────────────────────────────────
if np.any(conf_v):
    plot_fire_overlay(
        cube_v["MWIR_BT"].filled(np.nan), conf_v,
        f"Confirmed Fires (n={int(np.sum(conf_v))}) — {ts_label_v}",
        frp_values=frp_v,
    )

# ── Natural colour composite ──────────────────────────────────────────────
if "NCC" in cube_v:
    plot_ncc(cube_v["NCC"], f"Natural Colour Composite — {ts_label_v}")

# ── FRP timeseries and cumulative FRE ────────────────────────────────────
#   plot_frp_timeseries(timestamps, frp_total, frp_biebrza)
#   plot_cumulative_frp(timestamps, frp_biebrza)
plot_frp_timeseries(timestamps, frp_total_ts, frp_biebrza_ts)
plot_cumulative_frp(timestamps, frp_biebrza_ts)

In [ ]:
# ========================================================================
# Step 8: Interactive fire map
# ========================================================================
# build_interactive_map(gdf_all, BIEBRZA_BBOX) returns a leafmap.Map object.
# Serialize Timestamp columns to strings before calling add_gdf.

if not gdf_all.empty:
    import pandas as pd

    def _serialize(gdf):
        gdf = gdf.copy()
        for col in gdf.columns:
            if col == "geometry": continue
            if pd.api.types.is_datetime64_any_dtype(gdf[col]):
                gdf[col] = gdf[col].astype(str)
            elif gdf[col].dtype == object and len(gdf) > 0:
                s = gdf[col].dropna()
                if len(s) > 0 and hasattr(s.iloc[0], "strftime"):
                    gdf[col] = gdf[col].apply(lambda x: str(x) if x is not None else x)
        return gdf

    # Build and display the map
    #   build_interactive_map(gdf_all, bbox) → leafmap.Map
    m = build_interactive_map(_serialize(gdf_all), BIEBRZA_BBOX)
    display(m)
else:
    print("No fire detections to display.")

In [ ]:
# ========================================================================
# Step 9: Export results to GeoJSON
# ========================================================================
# gdf_all.to_file(path) writes a GeoDataFrame to GeoJSON (or Shapefile).

if not gdf_all.empty:
    out_path = OUTPUT_DIR / "fire_detections_all_timestamps.geojson"
    gdf_all.to_file(out_path)  # gdf_all.to_file(path, driver="ESRI Shapefile") for .shp
    print(f"Saved {len(gdf_all)} fire detections to: {out_path}")

    print(f"\nSummary:")
    print(f"  Timestamps processed: {len(timestamps)}")
    print(f"  Total detections:     {len(gdf_all)}")
    print(f"  Peak scene FRP:       {max(frp_total_ts):.1f} MW")
    print(f"  Peak Biebrza FRP:     {max(frp_biebrza_ts):.1f} MW")
    for dt, ft, fb in zip(timestamps, frp_total_ts, frp_biebrza_ts):
        print(f"    {dt.strftime("%H:%M UTC")}: Total={ft:.1f} MW  Biebrza={fb:.1f} MW")
else:
    print("No fire detections to export.")

---

## Part 4: Validation & Reflection

### Sanity checks

- [ ] Biebrza fire clearly detected at **13:00 UTC** (MWIR BT ~325 K)
- [ ] Total fire count is reasonable (tens to low hundreds, not thousands)
- [ ] FRP values are positive and plausible (1–100 MW per pixel)
- [ ] Fire appears in the **eastern part** of the Biebrza bbox (~23°E, 53.5°N)
- [ ] Interactive map shows clustering near the known fire location

### Questions to discuss

1. **Earliest detection:** At what timestamp did you first detect it? Could you go earlier? What was the trade-off?
2. **False alarm rate:** How many fire pixels are outside Biebrza? Are they plausible?
3. **Cloud mask sensitivity:** What happens when you change `nir13_thresh` from 0.005 to 0.02?
4. **Threshold curve:** Plot detections vs. MWIR threshold (300–320 K). What does it look like?
5. **Limitations:** What are the main limitations compared to operational systems?